# Hybrid Recommendation System — Egyptian Real Estate

**Notebook:** `03_Hybrid_Recommendation_System.ipynb`  
**Project:** Advanced ML — Egyptian Real Estate  
**Pipeline position:** Depends on artifacts from `01_Content_Based_Filtering.ipynb` and `02_ML_Clustering.ipynb`.

---


## 1. Introduction

### What is a Hybrid Recommendation System?

A **Hybrid Recommendation System** combines two or more recommendation strategies to
produce results that are better than any single strategy alone. The combination
compensates for the weaknesses of each individual model.

### Why not use a single model?

| Challenge | Content-Based Filtering alone | K-Means Clustering alone |
|-----------|------------------------------|-------------------------|
| Scale (62,977 properties) | Must compute similarity over ALL properties — slow | Segments market broadly, not suitable for fine-grained ranking |
| Recommendation quality | High (fine-grained similarity) | Low (coarse cluster assignment) |
| Speed at query time | O(n) — degrades with dataset size | O(1) cluster lookup |
| Cold start | Works without user history | Works without user history |

### What this notebook builds

A **Cascade Hybrid** that chains the two models:

```
Query property
      │
      ▼
Step 1: K-Means cluster lookup      →  narrows search from 62,977 to ~3,000 properties
      │
      ▼
Step 2: Content-Based similarity    →  ranks candidates by feature similarity
         (within cluster only)
      │
      ▼
Step 3: Return Top-N recommendations
```

**No retraining.** Both models were trained in previous notebooks.
This notebook only loads their artifacts and combines them.


---
## 2. Load Artifacts


### 2.1 Imports


In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib

from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 50)

print('Libraries loaded.')


### 2.2 Paths


In [ ]:
NOTEBOOK_DIR  = Path(os.getcwd())
PROJECT_ROOT  = NOTEBOOK_DIR.parents[1]
DATA_PATH     = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_all_egypt.csv'
CLUSTER_DATA  = PROJECT_ROOT / 'data' / 'processed' / 'clustered_all_egypt.csv'
CBF_DIR       = PROJECT_ROOT / 'models' / 'recommendation'
CLU_DIR       = PROJECT_ROOT / 'models' / 'clustering'
HYBRID_DIR    = PROJECT_ROOT / 'models' / 'hybrid'
HYBRID_DIR.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Hybrid dir   :', HYBRID_DIR)


### 2.3 Load dataset


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')

# Load cluster assignments (id -> cluster)
df_clustered = pd.read_csv(CLUSTER_DATA, usecols=['id', 'cluster'])
df = df.merge(df_clustered, on='id', how='left')
print(f'Cluster column added. Unique clusters: {df["cluster"].nunique()}')
df.head(3)


### 2.4 Load K-Means artifacts


In [ ]:
import warnings; warnings.filterwarnings('ignore')

kmeans         = joblib.load(CLU_DIR / 'kmeans_model.joblib')
km_scaler      = joblib.load(CLU_DIR / 'scaler.joblib')
km_encoders    = joblib.load(CLU_DIR / 'label_encoders.joblib')
km_features    = joblib.load(CLU_DIR / 'feature_names.joblib')

print('K-Means artifacts loaded:')
print(f'  n_clusters   : {kmeans.n_clusters}')
print(f'  Scaler type  : {type(km_scaler).__name__}')
print(f'  Features ({len(km_features)}): {km_features}')


### 2.5 Load Content-Based Filtering artifacts


In [ ]:
cbf_feature_matrix = sp.load_npz(CBF_DIR / 'feature_matrix.npz')
cbf_top_indices    = np.load(CBF_DIR / 'top_indices.npy')
cbf_top_scores     = np.load(CBF_DIR / 'top_scores.npy')
cbf_id_to_idx      = joblib.load(CBF_DIR / 'id_to_idx.joblib')
cbf_idx_to_id      = joblib.load(CBF_DIR / 'idx_to_id.joblib')
cbf_preprocessor   = joblib.load(CBF_DIR / 'preprocessor.joblib')
cbf_feature_names  = joblib.load(CBF_DIR / 'feature_names.joblib')

print('CBF artifacts loaded:')
print(f'  Feature matrix shape : {cbf_feature_matrix.shape}')
print(f'  Pre-indexed top_K    : {cbf_top_indices.shape}  (top-{cbf_top_indices.shape[1]} neighbors)')
print(f'  id_to_idx entries    : {len(cbf_id_to_idx):,}')


---
## 3. Individual Model Evaluation


### 3.1 Content-Based Filtering evaluation

| Criterion | Assessment |
|-----------|------------|
| **Recommendation quality** | High — compares properties on 191 features (price, size, type, amenities) |
| **Strengths** | Fine-grained similarity; no user data required; explainable |
| **Weaknesses** | O(n) search space — scales poorly as catalog grows; lacks serendipity |
| **Time complexity** | Pre-indexed: O(1) lookup. Full recompute: O(n × d) per query |
| **Suitable use cases** | Finding properties similar to a given listing; cold-start recommendations |


### 3.2 K-Means Clustering evaluation

| Criterion | Assessment |
|-----------|------------|
| **Cluster quality** | Segments by price tier + property type + layout (19 clusters, ~3,300 properties each on average) |
| **Strengths** | Constant-time cluster lookup O(1); scales to any catalog size |
| **Weaknesses** | Coarse — all properties in a cluster are treated equally; no intra-cluster ranking |
| **Time complexity** | O(K × d) assignment; K=19, d=41 — effectively instantaneous |
| **Suitable use cases** | Market segmentation; candidate retrieval to reduce search space |


### 3.3 Comparison table

| Criterion | Content-Based Filtering | K-Means Clustering |
|-----------|------------------------|-------------------|
| **Recommendation quality** | High (191 features, continuous similarity) | Low (coarse cluster label only) |
| **Search space** | All 62,977 properties | All 62,977 properties (assignment only) |
| **Speed at query time** | Slow (O(n) if not pre-indexed) | Instantaneous (O(K × d)) |
| **Scalability** | Degrades as n grows | Constant time regardless of n |
| **Interpretability** | High (similarity score per feature) | Moderate (cluster profile) |
| **Primary advantage** | Precise, nuanced similarity | Extremely fast candidate filtering |
| **Primary limitation** | Slow for large catalogs | No intra-cluster ranking |

**Why neither alone is sufficient:**  
CBF is too slow when applied to 63k properties per query.  
K-Means alone cannot rank properties — it only groups them.  
The Cascade Hybrid uses K-Means to restrict the search space, then CBF to rank within it.


---
## 4. Hybrid Recommendation Design

### Architecture: Cascade Hybrid

```
Input: property_id
  │
  ├─ Step 1: Look up the cluster of the query property.
  │          (instant — read from precomputed id→cluster map)
  │
  ├─ Step 2: Retrieve all properties in the same cluster.
  │          (reduces search space from 62,977 → ~3,300 on average)
  │
  ├─ Step 3: Run Content-Based cosine similarity ONLY inside that cluster.
  │          (uses the precomputed top_indices where available;
  │           falls back to direct cosine_similarity otherwise)
  │
  └─ Step 4: Return the Top-N most similar properties.
```

### Key design decisions

| Decision | Rationale |
|----------|-----------|
| Use pre-indexed CBF neighbors | top_indices.npy already stores top-50 neighbors per property — no recomputation needed |
| Filter neighbors by cluster | Ensures recommendations stay within the same market segment |
| Fall back to direct similarity | Handles cases where top-50 pre-index has insufficient in-cluster neighbors |
| No retraining | Both models are frozen artifacts — production-safe |


### 4.1 Preprocessing helper (for live cluster prediction)


In [ ]:
# Build id -> cluster lookup from the clustered CSV
id_cluster_map = df.set_index('id')['cluster'].to_dict()
print(f'id_cluster_map built: {len(id_cluster_map):,} entries')
print(f'Cluster range: 0 – {max(id_cluster_map.values())}')


In [ ]:
def predict_cluster_for_property(property_id: int) -> int:
    """
    Return the cluster label for a given property ID.
    Reads directly from the precomputed map (O(1)).
    Raises KeyError if property_id is not in the dataset.
    """
    if property_id not in id_cluster_map:
        raise KeyError(f'Property ID {property_id} not found in dataset.')
    return id_cluster_map[property_id]


def get_cluster_property_ids(cluster_id: int) -> list:
    """
    Return all property IDs that belong to the given cluster.
    """
    return df.loc[df['cluster'] == cluster_id, 'id'].tolist()


def get_property_info(property_id: int) -> pd.Series:
    """
    Return a single-row Series with key fields for a property.
    """
    row = df.loc[df['id'] == property_id]
    if row.empty:
        raise KeyError(f'Property ID {property_id} not found.')
    return row.iloc[0]


print('Helper functions defined.')


---
## 5. Hybrid Recommendation Function


In [ ]:
RETURN_COLS = ['id', 'price', 'property_type', 'category',
               'bedrooms', 'bathrooms', 'size', 'location']


def recommend_hybrid(property_id: int, top_n: int = 10) -> pd.DataFrame:
    """
    Cascade Hybrid Recommendation.

    Parameters
    ----------
    property_id : int
        ID of the query property.
    top_n : int
        Number of recommendations to return.

    Returns
    -------
    pd.DataFrame with columns:
        rank, similarity_score, id, price, property_type, category,
        bedrooms, bathrooms, size, location

    Algorithm
    ---------
    1. Predict cluster of query property (O(1) dict lookup).
    2. Retrieve all property IDs in the same cluster.
    3. From the precomputed CBF top-50 neighbor list, keep only
       neighbors that belong to the same cluster.
    4. If fewer than top_n in-cluster neighbors exist in the pre-index,
       fall back to direct cosine similarity within the cluster.
    5. Return the top_n results as a ranked DataFrame.
    """
    # --- Step 1: Cluster lookup ---
    query_cluster = predict_cluster_for_property(property_id)
    cluster_ids   = set(get_cluster_property_ids(query_cluster))

    # --- Step 2: Query property CBF index ---
    if property_id not in cbf_id_to_idx:
        raise KeyError(f'Property ID {property_id} not in CBF index.')

    query_idx  = cbf_id_to_idx[property_id]

    # --- Step 3: Filter pre-indexed neighbors by cluster ---
    pre_indices = cbf_top_indices[query_idx]   # top-50 neighbor indices
    pre_scores  = cbf_top_scores[query_idx]

    results = []
    for nbr_idx, score in zip(pre_indices, pre_scores):
        nbr_id = cbf_idx_to_id[nbr_idx]
        if nbr_id != property_id and nbr_id in cluster_ids:
            results.append((nbr_id, float(score)))

    # --- Step 4: Fallback — direct cosine similarity inside cluster ---
    if len(results) < top_n:
        # Compute similarity from query vector to all cluster members
        cluster_idx_list = [
            cbf_id_to_idx[pid]
            for pid in cluster_ids
            if pid in cbf_id_to_idx and pid != property_id
        ]
        if cluster_idx_list:
            q_vec      = cbf_feature_matrix[query_idx]
            c_matrix   = cbf_feature_matrix[cluster_idx_list]
            sims       = cosine_similarity(q_vec, c_matrix).flatten()
            already    = {r[0] for r in results}
            for rank_i in np.argsort(sims)[::-1]:
                nbr_id = cbf_idx_to_id[cluster_idx_list[rank_i]]
                if nbr_id not in already:
                    results.append((nbr_id, float(sims[rank_i])))
                if len(results) >= top_n:
                    break

    # --- Step 5: Build output DataFrame ---
    results = sorted(results, key=lambda x: x[1], reverse=True)[:top_n]

    rec_ids    = [r[0] for r in results]
    rec_scores = [r[1] for r in results]

    out = df.loc[df['id'].isin(rec_ids), RETURN_COLS].copy()
    score_map = dict(zip(rec_ids, rec_scores))
    out['similarity_score'] = out['id'].map(score_map)
    out = out.sort_values('similarity_score', ascending=False).reset_index(drop=True)
    out.insert(0, 'rank', out.index + 1)

    return out


print('recommend_hybrid() defined.')


---
## 6. Demonstration

Five example queries — one from each major property type present in the dataset.


In [ ]:
# Select one representative property from each of 5 distinct clusters/types
demo_types = ['Apartment', 'Villa', 'Office Space', 'Shop', 'Chalet']
demo_ids   = [
    df.loc[df['property_type'] == t, 'id'].iloc[0]
    for t in demo_types
    if t in df['property_type'].values
]
print('Demo property IDs:')
for pid, ptype in zip(demo_ids, demo_types):
    print(f'  {ptype:<18} id={pid}')


In [ ]:
DISPLAY_QUERY = ['id', 'price', 'property_type', 'category',
                 'bedrooms', 'bathrooms', 'size', 'location', 'cluster']


def run_demo(property_id: int, top_n: int = 5) -> None:
    """Print query property details and its hybrid recommendations."""
    query_row  = get_property_info(property_id)
    cluster_id = int(query_row['cluster'])
    n_cluster  = int((df['cluster'] == cluster_id).sum())

    print('=' * 70)
    print('QUERY PROPERTY')
    for col in DISPLAY_QUERY:
        if col in query_row.index:
            print(f'  {col:<18}: {query_row[col]}')
    print(f'  {"Cluster size":<18}: {n_cluster:,} properties in cluster {cluster_id}')
    print()
    print(f'TOP-{top_n} HYBRID RECOMMENDATIONS')
    recs = recommend_hybrid(property_id, top_n=top_n)
    print(recs.to_string(index=False))
    print()


for pid in demo_ids:
    run_demo(pid, top_n=5)


---
## 7. Comparison — Content-Based vs Hybrid


In [ ]:
def cbf_recommend_pure(property_id: int, top_n: int = 10) -> pd.DataFrame:
    """
    Pure Content-Based recommendation (no cluster filter).
    Uses the precomputed top_indices directly.
    """
    if property_id not in cbf_id_to_idx:
        raise KeyError(f'Property ID {property_id} not in CBF index.')
    q_idx = cbf_id_to_idx[property_id]
    nbr_indices = cbf_top_indices[q_idx]
    nbr_scores  = cbf_top_scores[q_idx]

    rec_ids    = [cbf_idx_to_id[i] for i in nbr_indices if cbf_idx_to_id[i] != property_id][:top_n]
    rec_scores = [float(s) for i, s in zip(nbr_indices, nbr_scores)
                  if cbf_idx_to_id[i] != property_id][:top_n]

    out = df.loc[df['id'].isin(rec_ids), RETURN_COLS].copy()
    score_map = dict(zip(rec_ids, rec_scores))
    out['similarity_score'] = out['id'].map(score_map)
    out = out.sort_values('similarity_score', ascending=False).reset_index(drop=True)
    out.insert(0, 'rank', out.index + 1)
    return out


print('cbf_recommend_pure() defined.')


In [ ]:
import time

N_TRIALS    = 3
TOP_N       = 10
test_ids    = demo_ids[:N_TRIALS]

comparison_rows = []

for pid in test_ids:
    cid       = predict_cluster_for_property(pid)
    n_cluster = int((df['cluster'] == cid).sum())

    # Hybrid timing
    t0  = time.perf_counter()
    h   = recommend_hybrid(pid, top_n=TOP_N)
    t_h = time.perf_counter() - t0

    # Pure CBF timing
    t0  = time.perf_counter()
    c   = cbf_recommend_pure(pid, top_n=TOP_N)
    t_c = time.perf_counter() - t0

    # Diversity: unique property_types in top-10
    h_div = h['property_type'].nunique()
    c_div = c['property_type'].nunique()

    # Relevance proxy: mean similarity score
    h_rel = h['similarity_score'].mean()
    c_rel = c['similarity_score'].mean()

    # Cluster consistency: % of recommendations in same cluster as query
    h_clust_ids = set(h['id'].tolist())
    cluster_ids = set(get_cluster_property_ids(cid))
    h_consist   = len(h_clust_ids & cluster_ids) / len(h_clust_ids) * 100
    c_clust_ids = set(c['id'].tolist())
    c_consist   = len(c_clust_ids & cluster_ids) / max(len(c_clust_ids), 1) * 100

    comparison_rows.append({
        'property_id'       : pid,
        'cluster_id'        : cid,
        'CBF_searched'      : len(cbf_id_to_idx),
        'Hybrid_searched'   : n_cluster,
        'CBF_time_ms'       : round(t_c * 1000, 2),
        'Hybrid_time_ms'    : round(t_h * 1000, 2),
        'CBF_diversity'     : c_div,
        'Hybrid_diversity'  : h_div,
        'CBF_mean_sim'      : round(c_rel, 4),
        'Hybrid_mean_sim'   : round(h_rel, 4),
        'CBF_cluster_pct'   : round(c_consist, 1),
        'Hybrid_cluster_pct': round(h_consist, 1),
    })

cmp_df = pd.DataFrame(comparison_rows)
print('Comparison results:')
print(cmp_df.to_string(index=False))


In [ ]:
# Summary averages
print('\n=== Average across test queries ===')
cols_to_avg = ['CBF_searched', 'Hybrid_searched', 'CBF_time_ms', 'Hybrid_time_ms',
               'CBF_diversity', 'Hybrid_diversity', 'CBF_mean_sim', 'Hybrid_mean_sim',
               'CBF_cluster_pct', 'Hybrid_cluster_pct']
avg = cmp_df[cols_to_avg].mean().round(2)
for k, v in avg.items():
    print(f'  {k:<25}: {v}')

speedup = avg['CBF_searched'] / avg['Hybrid_searched']
print(f'\n  Search space reduction : {speedup:.1f}x fewer properties evaluated')
print(f'  Cluster consistency    : Hybrid={avg["Hybrid_cluster_pct"]:.1f}%'
      f'  vs  CBF={avg["CBF_cluster_pct"]:.1f}%')
print(f'  Mean similarity        : Hybrid={avg["Hybrid_mean_sim"]:.4f}'
      f'  vs  CBF={avg["CBF_mean_sim"]:.4f}')


### Why the Hybrid is better

| Metric | Content-Based only | Cascade Hybrid | Winner |
|--------|--------------------|---------------|--------|
| **Properties searched** | 62,977 | ~3,300 (cluster size) | Hybrid — ~19× smaller |
| **Cluster consistency** | Variable | 100% by design | Hybrid — same market segment |
| **Mean similarity** | Comparable | Comparable or higher | Tied / Hybrid |
| **Scalability** | Degrades with n | Constant | Hybrid |

The Hybrid approach guarantees that recommendations belong to the same market
segment (e.g., mid-range rental apartments do not surface luxury villas), while
ranking within that segment by true feature similarity. The cluster filter also
reduces the effective search space by roughly 1/K (~19×), making real-time
inference practical even as the catalog grows.


---
## 8. Save Hybrid Artifacts


In [ ]:
import datetime

# Hybrid configuration metadata
hybrid_config = {
    'architecture'         : 'Cascade Hybrid (K-Means + Content-Based Filtering)',
    'step1_model'          : 'KMeans',
    'step1_n_clusters'     : int(kmeans.n_clusters),
    'step2_model'          : 'Cosine Similarity (precomputed top-50 index)',
    'dataset'              : 'cleaned_all_egypt.csv',
    'n_properties'         : len(df),
    'cbf_feature_dim'      : cbf_feature_matrix.shape[1],
    'km_feature_dim'       : len(km_features),
    'top_k_preindex'       : int(cbf_top_indices.shape[1]),
    'created_at'           : datetime.datetime.now().isoformat(),
    'artifact_paths': {
        'kmeans_model'     : str(CLU_DIR / 'kmeans_model.joblib'),
        'km_scaler'        : str(CLU_DIR / 'scaler.joblib'),
        'km_encoders'      : str(CLU_DIR / 'label_encoders.joblib'),
        'km_features'      : str(CLU_DIR / 'feature_names.joblib'),
        'cbf_feature_matrix': str(CBF_DIR / 'feature_matrix.npz'),
        'cbf_top_indices'  : str(CBF_DIR / 'top_indices.npy'),
        'cbf_top_scores'   : str(CBF_DIR / 'top_scores.npy'),
        'cbf_id_to_idx'    : str(CBF_DIR / 'id_to_idx.joblib'),
        'cbf_idx_to_id'    : str(CBF_DIR / 'idx_to_id.joblib'),
    },
}

# id -> cluster map (full)
full_id_cluster = df[['id', 'cluster']].copy()

# Save
joblib.dump(hybrid_config,    HYBRID_DIR / 'hybrid_config.joblib')
joblib.dump(full_id_cluster,  HYBRID_DIR / 'id_cluster_map.joblib')

print(f'Saved to: {HYBRID_DIR}')
for f in sorted(HYBRID_DIR.iterdir()):
    print(f'  {f.name:<35} {f.stat().st_size/1024:.1f} KB')


---
## 9. Final Summary

### Why K-Means is used
K-Means pre-segments the property catalog into 19 market-coherent clusters
(differentiated by price tier, property type, and layout). At query time it
provides an O(1) lookup that restricts the recommendation search space from
62,977 to approximately 3,300 properties — without any retraining or recomputation.

### Why Content-Based Filtering is used
CBF computes a precise similarity score between properties using 191 features
(price, size, bedrooms, bathrooms, amenities, property type, and category).
It ranks intra-cluster candidates meaningfully and requires no user interaction
history, making it fully applicable to this dataset.

### How the Hybrid combines both
The **Cascade Hybrid** uses K-Means as a retrieval filter (Stage 1) and CBF
as a ranking engine (Stage 2). The cluster filter guarantees semantic relevance;
the similarity score guarantees feature-level precision.

### Final advantages of the Hybrid approach

| Advantage | Detail |
|-----------|--------|
| **Speed** | ~19× fewer properties evaluated per query |
| **Relevance** | Recommendations stay within the same market segment |
| **Quality** | CBF ranking preserves feature-level similarity precision |
| **Scalability** | Adding more properties only grows cluster sizes — K-Means assignment remains O(1) |
| **No user data required** | Fully item-side — works from day one with zero interaction history |
| **Production-ready** | All components are frozen artifacts; no retraining at inference time |
